In [1]:
from z3 import *


In [14]:
# 1.1 Node counter + depth
# A bit of recusion + learning Z3 apis

def count_nodes(expr):
    return 1 + sum(count_nodes(expr.arg(i)) for i in range(expr.num_args()))

def depth(expr):
    assert is_app(expr), "application expected"
    return depth_rec(expr, 0)

def depth_rec(expr, depth):
    if expr.num_args() == 0:
        return depth
    return max(depth_rec(expr.arg(i), depth + 1) for i in range(expr.num_args()))


x, y = Ints("x y")
f = Function("f", IntSort(), IntSort(), IntSort())
g = Function("g", IntSort(), IntSort())

n = f(f(g(x), g(g(x))), g(g(y)))
print(count_nodes(n))
print(depth(n))


10
4


In [6]:
# 1.2 Count kinds
# Again some recursion

def kind_counts(expr):
    return kind_counts_rec(expr, {})

def kind_counts_rec(expr, counts):
    name = expr.decl().name()
    counts[name] = counts.get(name, 0) + 1
    for i in range(expr.num_args()):
        kind_counts_rec(expr.arg(i), counts)
    return counts

x, y = Ints("x y")
print(kind_counts(x*2 + y*(x*2) + 3))


{'+': 2, '*': 3, 'x': 2, 'Int': 3, 'y': 1}


In [5]:
# 2.1 Self inverse permutation for 4 elements 0..3
# I remember some leet code tasks where we had to find a cycle this way, why not trying to generate it with Z3

A = Array("A", IntSort(), IntSort())
cells = [Select(A, k) for k in range(4)]

solver = Solver()
solver.add(Distinct(cells))
solver.add(And([And(cell >= 0, cell <= 3) for cell in cells]))
solver.add(And([Select(A, Select(A, k)) == k for k in range(4)]))
solver.add(And([Select(A, k) != k for k in range(4)]))

print(solver.check())
print([solver.model().eval(Select(A, k)) for k in range(4)])


sat
[3, 2, 1, 0]


In [10]:
# 2.2 5 lamps in a row, find out whether it is possible to press a combination so that all of them are off, each press toggles i-1, i and i+1
# As it turned out not all permutations have a solution

press = Array("press", IntSort(), BoolSort())
start = [True, False, True, False, True]

def final_state(i):
    state = BoolVal(start[i])
    for j in [i - 1, i, i + 1]:
        if 0 <= j < 5:
            state = Xor(state, press[j])
    return state

solver = Solver()
for i in range(5):
    solver.add(Not(final_state(i)))

print(solver.check())


sat


In [11]:
# 3.1 Given an unkonw integer function with f(0) = 2 and f(x + 1) = f(x) + 3 show that f(5) == 17 follows.

f = Function("f", IntSort(), IntSort())
x = Int("x")

axioms = [ForAll(x, f(x + 1) == f(x) + 3), f(0) == 2]

solver = Solver()
solver.add(axioms)
solver.add(f(5) != 17)
print(solver.check())


unsat


In [12]:
# 3.2 The barber paradox in the village of barbers who shave exactly those villagers who do not shave themselves.

Person = DeclareSort("Person")
shaves = Function("shaves", Person, Person, BoolSort())
barber = Const("barber", Person)
p = Const("p", Person)

solver = Solver()
solver.add(ForAll(p, shaves(barber, p) == Not(shaves(p, p))))
print(solver.check())


unsat


In [13]:
# 4.1 Coin change
# Make exactly N coins of 1,5,10 and 25 with as few coins as possible. Now I can open my vending machine!

def minimize_coins(n):
    ones, fives, tens, quarters = Ints("ones fives tens quarters")
    opt = Optimize()
    opt.add(ones >= 0, fives >= 0, tens >= 0, quarters >= 0)
    opt.add(ones + 5 * fives + 10 * tens + 25 * quarters == n)
    opt.minimize(ones + fives + tens + quarters)

    if opt.check() != sat:
        return None

    model = opt.model()
    return {
        1: model[ones].as_long(),
        5: model[fives].as_long(),
        10: model[tens].as_long(),
        25: model[quarters].as_long(),
    }

print(minimize_coins(87))
print(minimize_coins(111))


{1: 2, 5: 0, 10: 1, 25: 3}
{1: 1, 5: 0, 10: 1, 25: 4}


In [8]:
# 4.2 Sell as much tables and chairs as possible
# Chair uses 2 units of woood + 3 hours of work and brings 40 profit. Table uses 5 wood and 4 hours of work + sells for 100 profit. Capacity: 40 wood and 30 hours Maximize profit!

chairs, tables = Ints("chairs tables")
profit = chairs * 40 + tables * 100

opt = Optimize()
opt.add(chairs >= 0, tables >= 0)
opt.add(chairs * 2 + tables * 5 <= 40)
opt.add(chairs * 3 + tables * 4 <= 30)
opt.maximize(profit)

print(opt.check())
print(opt.model())


sat
[chairs = 0, tables = 7]


In [2]:
# 5.1 All solutions for the rows of a magic square
# 1 <= x < y < z <= 9 and x + y + z == 15
# Actually quite nice that we can use the second solver to eliminate one solution, we could do it with the same solver of course, but for exercise purposes...

x, y, z = Ints("x y z")
base = Solver()
base.add(1 <= x, x < y, y < z, z <= 9, x + y + z == 15)

final = Solver()
final.add(base.assertions())

solutions = []
while final.check() == sat:
    model = final.model()
    solutions.append((model[x].as_long(), model[y].as_long(), model[z].as_long()))
    final.add(Or(x != model[x], y != model[y], z != model[z]))

print(solutions)


[(2, 4, 9), (3, 4, 8), (3, 5, 7), (4, 5, 6), (2, 6, 7), (2, 5, 8), (1, 5, 9), (1, 6, 8)]


In [9]:
# 5.2 Is an assumption redundant or contradictive by other assumptions. If none of them apply => informative

x, y = Ints("x y")
assumptions = [x > 0, y > x, y > 0, x + y < 10]

for i, assumption in enumerate(assumptions):
    others = assumptions[:i] + assumptions[i + 1:]

    redundant = Solver()
    redundant.add(others + [Not(assumption)])
    contradiction = Solver()
    contradiction.add(others + [assumption])

    if redundant.check() == unsat:
        verdict = "red"
    elif contradiction.check() == unsat:
        verdict = "contr"
    else:
        verdict = "info"

    print(assumption, verdict)


x > 0 informative
y > x informative
y > 0 redundant
x + y < 10 informative
